In [1]:
!pip install groq python-dotenv numpy tqdm datasets

In [2]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any

load_dotenv()
random.seed(0)

client = Groq()
gsm8k_dataset = load_dataset("gsm8k", "main")

gsm8k_train = gsm8k_dataset["train"]
gsm8k_test  = gsm8k_dataset["test"]

In [3]:
def generate_response_using_Llama(
        prompt: str,
        model: str = "llama3-8b-8192"
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.3, ### 수정해도 됩니다!
            stream=False
        )
        return chat_completion.choices[0].message.content
    
    except Exception as e:
        print(f"API call error: {str(e)}")
        return None

#### 응답 잘 나오는지 확인해보기

In [4]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello! I'm excited to help you with any math problems you'd like to solve. What kind of math are you working on? Do you have a specific problem in mind, or would you like me to suggest some examples to get us started?


#### GSM8K 데이터셋 확인해보기

In [5]:
print("[Question]")
for l in gsm8k_test['question'][0].split("."):
    print(l)
print("="*100)
print("[Answer]")
print(gsm8k_test['answer'][0])

[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


#### Util 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [6]:
### 수정해도 됩니다!
def extract_final_answer(response: str):
    regex = r"(?:Answer:|Model response:)\s*\$?([0-9,]+)\b|([0-9,]+)\s*(meters|cups|miles|minutes)"
    matches = re.finditer(regex, response, re.MULTILINE)
    results = [match.group(1) if match.group(1) else match.group(2).replace(",", "") for match in matches]

    if len(results) == 0:
        additional_regex = r"\$?([0-9,]+)"
        additional_matches = re.finditer(additional_regex, response, re.MULTILINE)
        results.extend([match.group(1).replace(",", "") for match in additional_matches])

    return results[-1] if results else None

In [7]:
### 수정해도 됩니다!
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama3-8b-8192",
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        correct_answer = float(re.findall(r'\d+(?:\.\d+)?', dataset[i]["answer"].split('####')[-1])[0])

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        if response:
            if VERBOSE:
                print("="*50)
                print(response)
                print("="*50)
            predicted_answer = extract_final_answer(response)

            if isinstance(predicted_answer, str):
                predicted_answer = float(predicted_answer.replace(",", ""))
            
            diff = abs(predicted_answer - correct_answer)
            is_correct = diff < 1e-5 if predicted_answer is not None else False
            
            if is_correct:
                correct += 1
            total += 1
            
            results.append({
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'response': response,
                'correct': is_correct
            })

            if (i + 1) % 5 == 0:
                current_acc = correct/total if total > 0 else 0
                print(f"Progress: [{i+1}/{num_samples}]")
                print(f"Current Acc.: [{current_acc:.2%}]")

    return results, correct/total if total > 0 else 0

In [8]:
def save_final_result(results: List[Dict[str, Any]], accuracy: float, filename: str) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += f"[Details]\n"
    
    for idx, result in enumerate(results):
        result_str += f"Question {idx+1}: {result['question']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)

#### Direct prompting with few-shot example

In [15]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.\n"

    for i in range(num_examples):
        cur_question = train_dataset['question'][i]
        cur_answer = train_dataset['answer'][i].split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [14]:
### 어떤 방식으로 저장되는지 확인해보세요!
PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")

 20%|██        | 2/10 [00:03<00:15,  1.98s/it]


KeyboardInterrupt: 

In [16]:
num_samples = 50

for shot in [0]:
    prompt = construct_direct_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        VERBOSE=False,
        num_samples=num_samples
    )
    save_final_result(results, accuracy, f"direct_prompting_{shot}.txt")

 10%|█         | 5/50 [00:05<00:51,  1.14s/it]

Progress: [5/50]
Current Acc.: [20.00%]


 20%|██        | 10/50 [00:09<00:35,  1.12it/s]

Progress: [10/50]
Current Acc.: [20.00%]


 30%|███       | 15/50 [00:14<00:30,  1.14it/s]

...

Progress: [40/50]
Current Acc.: [17.50%]


 90%|█████████ | 45/50 [01:03<00:15,  3.14s/it]

Progress: [45/50]
Current Acc.: [15.56%]


100%|██████████| 50/50 [01:15<00:00,  1.51s/it]

Progress: [50/50]
Current Acc.: [18.00%]


In [17]:
num_samples = 50

for shot in [3]:
    prompt = construct_direct_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        VERBOSE=False,
        num_samples=num_samples
    )
    save_final_result(results, accuracy, f"direct_prompting_{shot}.txt")

 10%|█         | 5/50 [00:09<01:37,  2.17s/it]

Progress: [5/50]
Current Acc.: [20.00%]


 20%|██        | 10/50 [00:22<01:45,  2.64s/it]

Progress: [10/50]
Current Acc.: [30.00%]


 30%|███       | 15/50 [00:33<01:27,  2.49s/it]

...

Progress: [40/50]
Current Acc.: [17.50%]


 90%|█████████ | 45/50 [01:54<00:13,  2.63s/it]

Progress: [45/50]
Current Acc.: [17.78%]


100%|██████████| 50/50 [02:07<00:00,  2.55s/it]

Progress: [50/50]
Current Acc.: [20.00%]


In [18]:
num_samples = 50

for shot in [5]:
    prompt = construct_direct_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        VERBOSE=False,
        num_samples=num_samples
    )
    save_final_result(results, accuracy, f"d_irect_prompting_{shot}.txt")

 10%|█         | 5/50 [00:02<00:23,  1.90it/s]

Progress: [5/50]
Current Acc.: [40.00%]


 20%|██        | 10/50 [00:05<00:22,  1.75it/s]

Progress: [10/50]
Current Acc.: [20.00%]


 30%|███       | 15/50 [00:08<00:19,  1.78it/s]

...

Progress: [40/50]
Current Acc.: [22.50%]


 90%|█████████ | 45/50 [01:48<00:18,  3.76s/it]

Progress: [45/50]
Current Acc.: [20.00%]


100%|██████████| 50/50 [02:09<00:00,  2.58s/it]

Progress: [50/50]
Current Acc.: [22.00%]


### Chain-of-Thought prompting with few-shot example
```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 되겠죠?

In [10]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question step-by-step, showing your reasoning, "
        "and finally provide the answer after the tag 'Answer:'.\n"
    )

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset['question'][i]
        # CoT 답변: rationale과 답을 포함하는 문자열
        cur_answer = train_dataset['answer'][i].strip()

        prompt += f"\n[Example {idx+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"{cur_answer}\n"  # 답변 안에 reasoning + Answer 포함되어 있다고 가정

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt


In [17]:
num_samples = 50

for shot in [0]:
    prompt = construct_CoT_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        VERBOSE=False,
        num_samples=num_samples
    )
    save_final_result(results, accuracy, f"CoT_prompting_{shot}.txt")

 10%|█         | 5/50 [00:22<03:21,  4.47s/it]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [00:53<03:35,  5.38s/it]

Progress: [10/50]
Current Acc.: [80.00%]


 30%|███       | 15/50 [00:59<01:06,  1.91s/it]

...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [01:54<00:12,  2.43s/it]

Progress: [45/50]
Current Acc.: [77.78%]


100%|██████████| 50/50 [02:14<00:00,  2.68s/it]

Progress: [50/50]
Current Acc.: [76.00%]


In [19]:
num_samples = 50

for shot in [3]:
    prompt = construct_CoT_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        VERBOSE=False,
        num_samples=num_samples
    )
    save_final_result(results, accuracy, f"CoT_prompting_{shot}.txt")

  0%|          | 0/50 [00:00<?, ?it/s]

 10%|█         | 5/50 [00:04<00:39,  1.14it/s]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [00:41<05:06,  7.65s/it]

Progress: [10/50]
Current Acc.: [80.00%]


...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [06:06<00:45,  9.13s/it]

Progress: [45/50]
Current Acc.: [77.78%]


100%|██████████| 50/50 [06:54<00:00,  8.30s/it]

Progress: [50/50]
Current Acc.: [80.00%]


In [21]:
num_samples = 50

for shot in [5]:
    prompt = construct_CoT_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        VERBOSE=False,
        num_samples=num_samples
    )
    save_final_result(results, accuracy, f"CoT_prompting_{shot}.txt")

  0%|          | 0/50 [00:00<?, ?it/s]

 10%|█         | 5/50 [00:07<01:03,  1.42s/it]

Progress: [5/50]
Current Acc.: [60.00%]


 20%|██        | 10/50 [00:54<06:16,  9.41s/it]

Progress: [10/50]
Current Acc.: [60.00%]


...

Progress: [40/50]
Current Acc.: [67.50%]


 90%|█████████ | 45/50 [07:25<00:56, 11.39s/it]

Progress: [45/50]
Current Acc.: [71.11%]


100%|██████████| 50/50 [08:21<00:00, 10.02s/it]

Progress: [50/50]
Current Acc.: [72.00%]


### Construct your prompt!!

목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올려보기!
- gsm8k의 train 데이터셋에서 예시를 가져온 다음 (자유롭게!)
- 그 예시들에 대한 풀이 과정을 만들어주세요!
- 모든 것들이 자유입니다! Direct Prompting, CoT Prompting을 한 결과보다 정답률만 높으면 돼요.

In [11]:
def construct_my_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train
    
    # 랜덤 샘플링 (올바른 방식)
    sampled_indices = random.sample(
        range(len(train_dataset['question'])), 
        num_examples
    )
    
    # 더 강력한 프롬프트 구성
    prompt = (
        "You are an expert mathematician with exceptional problem-solving skills. "
        "Your task is to solve mathematical word problems with the highest accuracy possible.\n\n"
        "IMPORTANT INSTRUCTIONS:\n"
        "1. Read the problem carefully and identify all given information\n"
        "2. Break down the problem into smaller, manageable steps\n"
        "3. Show your reasoning clearly and logically\n"
        "4. Double-check your calculations at each step\n"
        "5. Ensure your final answer is correct and complete\n"
        "6. Always end with 'Answer: [number]' format\n\n"
    )
    
    if num_examples > 0:
        prompt += "Here are some examples of how to solve similar problems:\n\n"
        
        for i in range(num_examples):
            cur_question = train_dataset['question'][sampled_indices[i]]
            cur_answer = train_dataset['answer'][sampled_indices[i]].strip()
            
            prompt += f"EXAMPLE {i+1}:\n"
            prompt += f"Problem: {cur_question}\n"
            prompt += f"Solution: {cur_answer}\n\n"

    prompt += (
        "Now solve the following problem using the same careful approach:\n"
        "Problem: {question}\n"
        "Solution:"
    )
    
    return prompt

In [23]:
# 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!
num_samples = 50

# My Prompting 테스트 실행
for shot in [0]:
    print(f"Running My Prompting with {shot} shots...")
    prompt = construct_my_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        VERBOSE=False,
        num_samples=num_samples
    )
    save_final_result(results, accuracy, f"My_prompting_{shot}.txt")
    print(f"My Prompting {shot}-shot accuracy: {accuracy:.2%}")
    print(f"Results saved to My_prompting_{shot}.txt")
    print("-" * 50)

Running My Prompting with 0 shots...


 10%|█         | 5/50 [00:06<01:02,  1.38s/it]

Progress: [5/50]
Current Acc.: [40.00%]


 20%|██        | 10/50 [00:24<02:29,  3.73s/it]

Progress: [10/50]
Current Acc.: [50.00%]


...

Progress: [40/50]
Current Acc.: [70.00%]


 90%|█████████ | 45/50 [03:10<00:17,  3.43s/it]

Progress: [45/50]
Current Acc.: [73.33%]


100%|██████████| 50/50 [03:33<00:00,  4.27s/it]

Progress: [50/50]
Current Acc.: [74.00%]
My Prompting 0-shot accuracy: 74.00%
Results saved to My_prompting_0.txt
--------------------------------------------------


In [24]:
# 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!
num_samples = 50

# My Prompting 테스트 실행
for shot in [3]:
    print(f"Running My Prompting with {shot} shots...")
    prompt = construct_my_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        VERBOSE=False,
        num_samples=num_samples
    )
    save_final_result(results, accuracy, f"My_prompting_{shot}.txt")
    print(f"My Prompting {shot}-shot accuracy: {accuracy:.2%}")
    print(f"Results saved to My_prompting_{shot}.txt")
    print("-" * 50)

Running My Prompting with 3 shots...


 10%|█         | 5/50 [00:33<06:04,  8.09s/it]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [01:16<05:49,  8.75s/it]

Progress: [10/50]
Current Acc.: [80.00%]


...

Progress: [40/50]
Current Acc.: [80.00%]


 90%|█████████ | 45/50 [06:14<00:42,  8.47s/it]

Progress: [45/50]
Current Acc.: [80.00%]


100%|██████████| 50/50 [06:57<00:00,  8.35s/it]

Progress: [50/50]
Current Acc.: [78.00%]
My Prompting 3-shot accuracy: 78.00%
Results saved to My_prompting_3.txt
--------------------------------------------------


In [13]:
# 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!
num_samples = 50

# My Prompting 테스트 실행
for shot in [5]:
    print(f"Running My Prompting with {shot} shots...")
    prompt = construct_my_prompt(shot)
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=prompt,
        VERBOSE=False,
        num_samples=num_samples
    )
    save_final_result(results, accuracy, f"My_prompting_{shot}.txt")
    print(f"My Prompting {shot}-shot accuracy: {accuracy:.2%}")
    print(f"Results saved to My_prompting_{shot}.txt")
    print("-" * 50)

Running My Prompting with 5 shots...


  0%|          | 0/50 [00:00<?, ?it/s]

 10%|█         | 5/50 [00:53<09:20, 12.45s/it]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [02:04<09:34, 14.37s/it]

...

Progress: [40/50]
Current Acc.: [80.00%]


 90%|█████████ | 45/50 [10:06<01:08, 13.67s/it]

Progress: [45/50]
Current Acc.: [82.22%]


100%|██████████| 50/50 [11:17<00:00, 13.55s/it]

Progress: [50/50]
Current Acc.: [84.00%]
My Prompting 5-shot accuracy: 84.00%
Results saved to My_prompting_5.txt
--------------------------------------------------


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot, 5 shot 정답률을 표로 보여주세요!
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요!
3. 본인이 작성한 프롬프트 기법이 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요!
4. 최종적으로, `PROMPTING.md`에 보고서를 작성해주세요!